<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Homework/HW06-supervised_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 6: Build Your Own Spam Filter

**DSC 210 Foundations of Data Science**

*Covers Module 10 (supervised learning: classification).*

```
ASK  ->  GET  ->  EXPLORE  ->  [ MODEL ]  ->  COMMUNICATE
```

References:
- Module 10 course notebook (train/test split, k-nearest-neighbors, decision trees, random forests, confusion matrices)
- Spam email data: TidyTuesday, 2023-08-15 ([data and dictionary](https://github.com/rfordatascience/tidytuesday/tree/main/data/2023/2023-08-15)), taken from the `spam7` data in the R package DAAG ([documentation](https://vincentarelbundock.github.io/Rdatasets/doc/DAAG/spam7.html))
- The original data: Hopkins, M., Reeber, E., Forman, G., and Suermondt, J. (1999). *Spambase*. UCI Machine Learning Repository. https://archive.ics.uci.edu/dataset/94/spambase

*Last major revision: 2026-11-03*

## Key Concepts

- Split data into training and test sets, and keep the test set out of every decision
- Beat a baseline, and report every accuracy next to one
- Build and compare k-nearest-neighbors, a decision tree, and a random forest
- Compute a Gini gain by hand, and read the rule a tree learned
- Read a confusion matrix, and choose between models using precision and recall
- Explain what a filter can and cannot learn from the features it is given

## Instructions

**Total: 50 points.** Point values are listed with each problem.

- Cells marked `# RUN-TOGETHER` are complete; run them and read the output.
- Cells marked `# FILL-IN` contain blanks written as `____`. They raise an error until every blank is replaced. Lines marked `# YOUR CODE` are yours to write.
- Run the cells **in order**. Later problems use the training and test sets from Part A.
- Every figure must include a title and labeled axes.
- Most of the credit rides on your written answers. Quote specific numbers from your output.
- Submit the completed `.ipynb`, with all cells run and all figures visible, to Canvas.

---
## Part 0. The assignment and the data (2 points)
---

**The scenario.** You have joined a small company that is writing its own email program, and your first job is to build its **spam filter, from scratch**. You will build it in versions, each one smarter than the last, and test every version the same way:

| version | how it decides |
| --- | --- |
| **v0** | never calls anything spam (this is the baseline every filter must beat) |
| **v1** | a rule you write by hand |
| **v2** | k-nearest-neighbors |
| **v3** | a decision tree |
| **v4** | a random forest |

At the end you will choose one version to ship, and then send it emails you write yourself.

**About the data.** The filter learns from 4,601 emails collected at Hewlett-Packard Labs in 1999, each labeled spam or not spam by the people who received it. The text of the emails is gone and each email is described by six numbers:

| Column | Meaning |
| --- | --- |
| `crl.tot` | total number of capital letters in uninterrupted runs of capitals (for example, "FREE MONEY" contributes 9) |
| `dollar` | `$` signs, as a percentage of all characters in the email |
| `bang` | `!` marks, as a percentage of all characters |
| `money` | the word "money", as a percentage of all words |
| `n000` | the string "000" (as in $1,000,000), as a percentage of all words |
| `make` | the word "make", as a percentage of all words |
| `yesno` | the label: `y` spam, `n` not spam |

In [ ]:
# RUN-TOGETHER  (setup)
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2023/2023-08-15/spam.csv'
spam = pd.read_csv(url)
features = ['crl.tot', 'dollar', 'bang', 'money', 'n000', 'make']

print('shape:', spam.shape)
print(spam['yesno'].value_counts(normalize=True).round(3))
spam.head()

**Problem 0.** (2 points)

(a) What does one row of `spam` represent? What information about the email is not in the row?

(b) What fraction of the emails are spam? Filter v0 calls every email "not spam". What accuracy will v0 score, and how much spam will it catch?

**Your answer:**

(a)

(b)

---
## Part A. Split, then the first two filters (12 points)
---

### Problem A1. Split before you build (3 points)

Every version of the filter will be built on the **training** emails and tested on the **test** emails, which no version is allowed to see while it is being built.

In [ ]:
# FILL-IN  (Problem A1)
X = spam[features]
y = spam['yesno']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=____, random_state=210, stratify=____)

print('training emails:', len(X_train), '| test emails:', len(X_test))
print('proportion spam, training:', round((y_train == 'y').mean(), 3))
print('proportion spam, test    :', round((y_test == 'y').mean(), 3))

# Filter v0: call everything 'n'. Its accuracy on the test emails:
print('v0 test accuracy    :', round((y_test == 'n').mean(), 3))

Use a quarter of the emails for testing, and keep the spam share the same in both halves.

(a) How many emails are in each set? What does `stratify` guarantee, and why does it matter for a filter like v0?

(b) The line for v0 computes `(y_test == 'n').mean()`. Explain why that is v0's accuracy.

**Your answer:**

(a)

(b)

### Problem A2. Filter v1: write your own rule (3 points)

Before any machine learning, write a filter by hand. Look at how spam and non-spam differ **in the training emails only**.

In [ ]:
# RUN-TOGETHER  (Problem A2): compare the two kinds of email, training data only
train = X_train.copy()
train['yesno'] = y_train
print('medians')
print(train.groupby('yesno')[features].median())
print()
print('means')
print(train.groupby('yesno')[features].mean().round(3))

Choose **two** features and a threshold for each, and call an email spam if **either** one is above its threshold. For example, `(X_test['bang'] > 1) | (X_test['money'] > 1)` would be one such rule (not a good one).

In [ ]:
# FILL-IN  (Problem A2): your rule, tested on the test emails
v1_says_spam = (X_test[____] > ____) | (X_test[____] > ____)

is_spam = (y_test == 'y')
print('v1 test accuracy:', round((v1_says_spam == is_spam).mean(), 3))
print('proportion of the test spam that v1 catches:', round(v1_says_spam[is_spam].mean(), 3))

(a) State your rule in words, and explain how the training medians or means guided your thresholds.

(b) Report v1's test accuracy and the share of spam it catches, and compare with v0.

**Your answer:**

(a)

(b)


### Problem A3. Filter v2: k-nearest-neighbors, and the ruler problem (3 points)

In [ ]:
# RUN-TOGETHER
print(X_train.describe().loc[['min', 'max']])

In [ ]:
# FILL-IN  (Problem A3)
scaler = StandardScaler().fit(____)          # learn the mean and SD from the TRAINING emails only
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(____)

knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
print('v2 (k-NN, k=5), scaled  :', round(knn.score(X_test_s, y_test), 3))

knn_raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
print('v2 (k-NN, k=5), unscaled:', round(knn_raw.score(X_test, y_test), 3))

(a) Which feature has by far the largest range? Explain why, without scaling, the "nearest" emails are mostly the ones with a similar value of that one feature.

(b) How much does scaling change v2's test accuracy? Does scaled v2 beat your v1?

**Your answer:**

(a)

(b)

### Problem A4. How many neighbors? (3 points)

In [ ]:
# RUN-TOGETHER
for k in [1, 5, 15, 51, 151]:
    model = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)
    print(f'k={k:4}   train={model.score(X_train_s, y_train):.3f}   test={model.score(X_test_s, y_test):.3f}')

(a) **(1 point)** Copy four numbers from the output into the table, then circle the row with the larger gap between training and test accuracy.

| `k` | training accuracy | test accuracy | gap (training minus test) |
| --- | --- | --- | --- |
| 1 |  |  |  |
| 5 |  |  |  |

(b) **(1 point)** Why is the training accuracy so high when `k = 1`? Choose one.

- **A.** `k = 1` is the best choice of `k` for this data.
- **B.** Each training email's nearest neighbor is itself, so on the training emails the filter is just looking up answers it has already memorized.
- **C.** The training emails are easier to classify than the test emails.
- **D.** Scaling only works when `k = 1`.

(c) **(1 point)** A classmate says: "I'll try every `k`, report whichever has the highest **test** accuracy, and that number is how well my filter will do on new email." What is the main problem with this plan? Choose one.

- **A.** There is no problem, because the test emails were never used to train the filter.
- **B.** Choosing `k` by its test score lets the test emails influence the filter, so the reported score is likely too optimistic.
- **C.** The highest test accuracy always happens at `k = 1`.
- **D.** Test accuracy cannot be computed for k-nearest-neighbors.

**Your answer:**

(a) Fill in the table above.

(b)

(c)

---
## Part B. Filter v3: a decision tree (12 points)
---

### Problem B1. One split, by hand (4 points)

Here are ten real emails from the data, sorted by `bang`.

In [ ]:
# RUN-TOGETHER
ten = spam.sample(10, random_state=13141).sort_values('bang')
ten[['bang', 'yesno']]

| `bang` | 0.000 | 0.016 | 0.027 | 0.048 | 0.361 | 0.488 | 0.834 | 0.875 | 0.952 | 1.289 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| label | n | n | n | n | n | y | y | y | n | y |

A tree considers two candidate questions: **Split 1: `bang` < 0.2** and **Split 2: `bang` < 0.4**.

(a) Compute the Gini index of the parent node (all ten emails).

(b) For each split, give the counts in each child, the Gini of each child, the weighted child Gini, and the Gini gain. Which split would the tree choose?

(c) Both splits produce a left child with Gini 0. Why does one still have a larger gain? Name the single email responsible.

(d) The email at `bang` = 0.952 is not spam even though it is full of exclamation marks. No threshold on `bang` alone can fix that. What could the tree do next?

Check your arithmetic with the cell below.

**Your answer:**

(a)

(b)

(c)

(d)

In [ ]:
# FILL-IN  (Problem B1): check your arithmetic
def gini(n_spam, n_not):
    n = n_spam + n_not
    if n == 0:
        return 0.0
    return 1 - ((n_spam / n) ** 2 + (n_not / n) ** 2)

parent = gini(4, 6)
print('parent Gini:', round(parent, 4))

# Split 1: bang < 0.2.  Fill in (number of spam, number not spam) for each child.
left1, right1 = gini(____, ____), gini(____, ____)
weighted1 = 4/10 * left1 + 6/10 * right1
print('split 1: left', round(left1, 4), ' right', round(right1, 4), ' weighted', round(weighted1, 4), ' gain', round(parent - weighted1, 4))

# Split 2: bang < 0.4
left2, right2 = gini(____, ____), gini(____, ____)
weighted2 = ____/10 * left2 + ____/10 * right2
print('split 2: left', round(left2, 4), ' right', round(right2, 4), ' weighted', round(weighted2, 4), ' gain', round(parent - weighted2, 4))

### Problem B2. Read the rule the tree learned (4 points)

Your v1 rule used thresholds you picked. A tree picks its own, by trying every threshold of every feature on the training emails and keeping the largest Gini gain.

In [ ]:
# FILL-IN  (Problem B2)
tree2 = DecisionTreeClassifier(max_depth=____, random_state=0).fit(X_train, y_train)
print('v3 (tree, depth 2) test accuracy:', round(tree2.score(X_test, y_test), 3))

fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(tree2, feature_names=features, class_names=tree2.classes_, filled=True, rounded=True, fontsize=10)
plt.title('Filter v3: a depth-2 decision tree')
plt.show()

(a) Write the tree as a rule in plain words, the way the Spam Filter Example in Module 10 wrote Tree A and Tree B. Which two features did it use, and how do they compare with the two you chose in A2?

(b) Run these two emails down the tree: email P has `bang` = 0.05 and `dollar` = 0.20; email Q has `bang` = 0.30 and `dollar` = 0.00. What does the tree call each one?

(c) Both leaves on the right side of the tree say spam. Use the `value` counts in those two leaves to explain why the tree still split that node.

**Your answer:**

(a)

(b)

(c)

### Problem B3. How deep should the tree grow? (4 points)

In [ ]:
# FILL-IN  (Problem B3)
for depth in [1, 2, 3, 5, 8, 12, None]:
    t = DecisionTreeClassifier(max_depth=____, random_state=0).fit(X_train, y_train)
    print(f'max_depth={str(depth):5}  train={t.score(X_train, y_train):.3f}  test={t.score(____, ____):.3f}')

Module 10 said a tree with no depth limit keeps splitting until every leaf is pure, which usually means 100% training accuracy. The cell below looks for a reason this tree falls short.

In [ ]:
# RUN-TOGETHER  (Problem B3): training emails whose six numbers are identical, but whose labels are not
groups = train.groupby(features)['yesno'].value_counts().unstack(fill_value=0)
mixed = groups[(groups['n'] > 0) & (groups['y'] > 0)]
print('groups of identical-looking training emails with both labels:', len(mixed))
print('training emails in those groups:', mixed.sum().sum())
mixed.sort_values('n', ascending=False).head()

(a) Describe what happens to training and test accuracy as `max_depth` grows. At about what depth does the gap between them start to widen? What is this called?

(b) Even with no depth limit, training accuracy stays below 1.000. Use the output of the second cell to explain why **no** filter built from these six numbers can be perfect, even on its own training emails. What do the largest look-alike groups have in common?

(c) In one or two sentences, explain **overfitting** in your own words, using the `max_depth=None` row as your example.

**Your answer:**

(a)

(b)

(c)

---
## Part C. Filter v4: a random forest (4 points)
---

In [ ]:
# FILL-IN  (Part C)
deep_tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=____, random_state=0).fit(X_train, y_train)

print(f'one deep tree :  train={deep_tree.score(X_train, y_train):.3f}  test={deep_tree.score(X_test, y_test):.3f}')
print(f'random forest :  train={forest.score(X_train, y_train):.3f}  test={forest.score(X_test, y_test):.3f}')

Use a forest of 200 trees.

(a) Both models fit the training emails equally well. Which does better on the test emails? Convert the difference in test accuracy into a number of **emails** out of 1,151.

(b) Using Section 5 of Module 10, explain in two or three sentences why many deep trees that vote can do better than one deep tree, and what we give up by using the forest.

**Your answer:**

(a)

(b)

---
## Part D. Which mistakes does each filter make? (12 points)
---

### Problem D1. Confusion matrices for v2, v3, and v4 (5 points)

For a spam filter, **spam is the positive class**: the thing we are trying to catch. We list `'y'` first so the tables match Module 10's layout (actual spam in the first row, predicted spam in the first column). For v3 we use a depth-3 tree.

In [ ]:
# FILL-IN  (Problem D1)
tree3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)

predictions = {
    'v2 k-NN (k=5)': knn.predict(X_test_s),
    'v3 tree (depth 3)': tree3.predict(X_test),
    'v4 forest': forest.predict(____),
}

order = ['y', 'n']    # spam first
for name, pred in predictions.items():
    print('=====', name, '=====')
    print(pd.DataFrame(confusion_matrix(y_test, pred, labels=order),
                       index=['actually spam', 'actually not spam'],
                       columns=['predicted spam', 'predicted not spam']))
    print(classification_report(y_test, pred, labels=____, digits=3))

(a) From the **forest's** confusion matrix, write down TP, FN, FP, and TN. Compute its spam precision and spam recall by hand, and check them against the `y` row of its report.

(b) In plain words, what does the forest's recall mean for a user's inbox? What does its precision mean?

(c) Make a small table of spam precision and spam recall for the three filters. Which filter misses the most spam? Which wrongly flags the fewest legitimate emails?

**Your answer:**

(a)

(b)

(c)

### Problem D2. Look at the mistakes (3 points)

Numbers like recall summarize the mistakes. Now look at them.

In [ ]:
# RUN-TOGETHER
results = X_test.copy()
results['actual'] = y_test
results['forest says'] = forest.predict(X_test)

missed = results[(results['actual'] == 'y') & (results['forest says'] == 'n')]   # spam that got through
flagged = results[(results['actual'] == 'n') & (results['forest says'] == 'y')]  # good email marked spam

print('spam that got through:', len(missed))
print(missed[features].median())
print()
print('good email marked as spam:', len(flagged))
print(flagged[features].median())

(a) Compare the medians of the spam that got through with the spam medians you saw in Problem A2. What do the missed spam emails look like?

(b) The good emails marked as spam: which feature looks most "spammy" about them?

(c) Connect (a) to Problem B3(b): why are these emails hard for **any** filter built on these six numbers?

**Your answer:**

(a)

(b)

(c)

### Problem D3. Code review: four quiet bugs (4 points)

A teammate wrote the cell below to report on the filters. **Every line runs.** For each numbered line, say what is wrong and why it matters, and whether the output gives the bug away. Then write a corrected version of each line.

In [ ]:
# RUN-TOGETHER  (a teammate's code: it runs, but every line has a problem)

# 1. Scale the data for k-NN
leaky = StandardScaler().fit(X)
knn_1 = KNeighborsClassifier(n_neighbors=5).fit(leaky.transform(X_train), y_train)
print('1. k-NN test accuracy:', round(knn_1.score(leaky.transform(X_test), y_test), 3))

# 2. How good is v2 on new email?
print('2. k-NN accuracy on new email:', round(knn.score(X_train_s, y_train), 3))

# 3. k-NN again
knn_3 = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
print('3. k-NN test accuracy:', round(knn_3.score(X_test_s, y_test), 3))

# 4. The forest's spam recall
cm = confusion_matrix(y_test, forest.predict(X_test))
tp, fn = cm[0, 0], cm[0, 1]
print('4. forest spam recall:', round(tp / (tp + fn), 3))

**Your answer:**

| Line | What is wrong, and why it matters | Does the output give it away? |
| --- | --- | --- |
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 4 |  |  |

In [ ]:
# Problem D3: your corrected lines here (label each one 1 to 4)

---
## Part E. Ship it (8 points)
---

### Problem E1. Send your filter some email (2 points)

The cell below turns the text of an email into the same six numbers the filter was trained on. (It follows the definitions in the data dictionary as closely as possible; you do not need to follow its code.) It then asks the forest and the depth-2 tree for their verdicts.

In [ ]:
# RUN-TOGETHER  (a helper: email text -> the six features)
import re

def email_features(text):
    words = [w.lower() for w in re.findall(r'[A-Za-z0-9]+', text)]
    n_words = max(len(words), 1)
    n_chars = max(len(text), 1)
    capital_runs = re.findall(r'[A-Z]+', text)
    row = {'crl.tot': max(sum(len(run) for run in capital_runs), 1),
           'dollar': 100 * text.count('$') / n_chars,
           'bang':   100 * text.count('!') / n_chars,
           'money':  100 * words.count('money') / n_words,
           'n000':   100 * words.count('000') / n_words,
           'make':   100 * words.count('make') / n_words}
    return pd.DataFrame([row])[features]

def check_email(text):
    row = email_features(text)
    print(row.round(3).to_string(index=False))
    print('forest says:', forest.predict(row)[0], '| depth-2 tree says:', tree2.predict(row)[0])
    print()

check_email("Hi Sam, are we still meeting at 3 to go over the lab report? Thanks, Alex")
check_email("CONGRATULATIONS!!! You have WON $1,000,000!!! Claim your MONEY now!!!")

Now write your own emails. Replace the two strings below: one ordinary email you might really send, and one spam email **designed to sneak past the filter**.

In [ ]:
# FILL-IN  (Problem E1)
check_email(____)      # an ordinary email
check_email(____)      # a spam email that tries to fool the filter

(a) Did both filters label your ordinary email correctly?

(b) Did your sneaky spam get through? Explain, using the six features, what you did to fool the filter (or why it did not work).

**Your answer:**

(a)

(b)

### Problem E2. Your recommendation (6 points)

Write for the company's product manager, who has never trained a model.

**(a) (3 points)** The company will use the filter in one of two ways: **delete** suspected spam without showing it to the user, or **tag** it with "possible spam" and leave it in the inbox. For each use, say which version (v0 to v4) you would ship and why. Quote precision, recall, or counts of mistakes from Part D.

**(b) (2 points)** Give two honest cautions about how well this filter will work on the company's real email.

**(c) (1 point)** Name one change to the **data** (not the algorithm) that would most likely make the filter better, and explain why.

**Your answer:**

(a)

(b)

(c)